In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip -q install -U transformers datasets accelerate sentencepiece wandb

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import wandb

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 75.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 54.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 24.0 MB/s eta 0:00:00
Device: cpu


In [3]:
wandb.login()

wandb.init(
    entity="mrinal-pandey2905-pes-university",
    project="23f2000333-t22026",
    name="milestone-5",
    config={
        "deberta":"microsoft/deberta-v3-small",
        "roberta":"roberta-base",
        "ensemble":"0.7/0.3",
        "tta":True
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [4]:
train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [5]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {
    0:"A",
    1:"B",
    2:"C",
    3:"D",
    4:"E"
}

train["label"] = train.answer.map(label2id)

In [6]:
DEBERTA_MODEL = "microsoft/deberta-v3-small"
ROBERTA_MODEL = "roberta-base"

In [7]:
deberta_tokenizer = AutoTokenizer.from_pretrained(
    DEBERTA_MODEL
)

roberta_tokenizer = AutoTokenizer.from_pretrained(
    ROBERTA_MODEL
)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
deberta = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL,
    num_labels=5,
    ignore_mismatched_sizes=True
).to(DEVICE)

roberta = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL,
    num_labels=5,
    ignore_mismatched_sizes=True
).to(DEVICE)

deberta.eval()
roberta.eval()

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifie

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [9]:
print(deberta.config)
print(roberta.config)

DebertaV2Config {
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "dtype": "float16",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4
  },
  "layer_norm_eps": 1e-07,
  "legacy": true,
  "max_position_embeddings": 512,
  "max_relative_positions": -1,
  "model_type": "deberta-v2",
  "norm_rel_ebd": "layer_norm",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "pooler_dropout": 0.0,
  "pooler_hidden_act": "gelu",
  "pooler_hidden_size": 768,
  "pos_att_type": [
    "p2c",
    "c2p"
  ],
  "position_biased_input": false,
  "position_buckets": 256,
  "relative_attention": true,
  "share_att_key": true,
  "tie_word_e

In [10]:
OPTIONS = ["A", "B", "C", "D", "E"]


def build_inputs(row, tta=False):
    """
    Create five prompt-option pairs for one MCQ.
    """

    prompt = str(row["prompt"])

    if tta:
        prompt = (
            "Answer the following multiple-choice question carefully:\n\n"
            + prompt
        )

    pairs = []

    for opt in OPTIONS:
        pairs.append((prompt, str(row[opt])))

    return pairs

In [11]:
def tokenize_mcq(tokenizer, row, tta=False, max_length=256):

    pairs = build_inputs(row, tta)

    prompts = [p[0] for p in pairs]
    answers = [p[1] for p in pairs]

    encoding = tokenizer(
        prompts,
        answers,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

    return {
        k: v.to(DEVICE)
        for k, v in encoding.items()
    }

In [12]:
@torch.no_grad()
def predict_probabilities(
    model,
    tokenizer,
    row,
    tta=False
):

    inputs = tokenize_mcq(
        tokenizer,
        row,
        tta=tta
    )

    outputs = model(**inputs)

    # logits shape = (5,5)
    logits = outputs.logits

    # Use diagonal 
    logits = torch.diagonal(logits)

    probs = F.softmax(
        logits,
        dim=0
    )

    return probs.cpu().numpy()

In [13]:
def topk_prediction(
    probabilities,
    k=3
):

    order = np.argsort(
        probabilities
    )[::-1]

    labels = [
        id2label[i]
        for i in order[:k]
    ]

    return labels

In [14]:
def weighted_ensemble(
    deberta_probs,
    roberta_probs,
    w1=0.7,
    w2=0.3
):

    return (
        w1 * deberta_probs +
        w2 * roberta_probs
    )

In [15]:
def average_ensemble(
    deberta_probs,
    roberta_probs
):

    return (
        deberta_probs +
        roberta_probs
    ) / 2

In [16]:
def tta_prediction(
    model,
    tokenizer,
    row
):

    p1 = predict_probabilities(
        model,
        tokenizer,
        row,
        tta=False
    )

    p2 = predict_probabilities(
        model,
        tokenizer,
        row,
        tta=True
    )

    return (p1 + p2) / 2

In [17]:
def apk(actual, predicted, k=3):

    score = 0
    hits = 0

    predicted = predicted[:k]

    for i, p in enumerate(predicted):

        if p == actual and p not in predicted[:i]:

            hits += 1

            score += hits / (i + 1)

    return score


def mapk(actuals, predictions):

    return np.mean([
        apk(a, p)
        for a, p in zip(actuals, predictions)
    ])

In [18]:
row = train.iloc[25]

d_probs = predict_probabilities(
    deberta,
    deberta_tokenizer,
    row
)

r_probs = predict_probabilities(
    roberta,
    roberta_tokenizer,
    row
)

print("DeBERTa")
print(d_probs)

print()

print("RoBERTa")
print(r_probs)

DeBERTa
[0.1873 0.237  0.1625 0.2074 0.2058]

RoBERTa
[0.19155662 0.20002307 0.21851256 0.1914907  0.19841702]


In [19]:
row = train.iloc[25]

In [20]:
deberta_probs = predict_probabilities(
    deberta,
    deberta_tokenizer,
    row
)

print("DeBERTa")

for option,prob in zip(
    OPTIONS,
    deberta_probs
):
    print(option,round(prob,6))

DeBERTa
A nan
B nan
C nan
D nan
E nan


/tmp/ipykernel_58/4045642318.py:13: RuntimeWarning: overflow encountered in cast
  print(option,round(prob,6))
/tmp/ipykernel_58/4045642318.py:13: RuntimeWarning: invalid value encountered in divide
  print(option,round(prob,6))


In [21]:
roberta_probs = predict_probabilities(
    roberta,
    roberta_tokenizer,
    row
)

print("RoBERTa")

for option,prob in zip(
    OPTIONS,
    roberta_probs
):
    print(option,round(prob,6))

RoBERTa
A 0.191557
B 0.200023
C 0.218513
D 0.191491
E 0.198417


In [22]:
q1_idx = np.argmax(
    deberta_probs
)

print("="*60)
print("Q1")
print(
    id2label[q1_idx],
    round(
        deberta_probs[q1_idx],
        4
    )
)

Q1
B 0.237


In [23]:
avg_probs = average_ensemble(
    deberta_probs,
    roberta_probs
)

q2_idx = np.argmax(
    avg_probs
)


print("Q2")

print(
    id2label[q2_idx]
)

Q2
B


In [24]:
weighted_probs = weighted_ensemble(
    deberta_probs,
    roberta_probs
)

q3_idx = np.argmax(
    weighted_probs
)



print("Q3")

print(
    id2label[q3_idx]
)

Q3
B


In [25]:
prediction = topk_prediction(
    weighted_probs,
    k=3
)

prediction = " ".join(
    prediction
)



print("Q4")

print(prediction)

Q4
B E D


In [26]:
predictions = []

for _, row in tqdm(test.iterrows(), total=len(test)):

    deberta_probs = predict_probabilities(
        deberta,
        deberta_tokenizer,
        row
    )

    roberta_probs = predict_probabilities(
        roberta,
        roberta_tokenizer,
        row
    )

    probs = weighted_ensemble(
        deberta_probs,
        roberta_probs,
        w1=0.7,
        w2=0.3
    )

    top3 = topk_prediction(
        probs,
        k=3
    )

    predictions.append(
        " ".join(top3)
    )

  0%|          | 0/500 [00:00<?, ?it/s]

In [27]:
submission = pd.DataFrame({

    "id":test.id,

    "prediction":predictions

})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,prediction
0,1,B E D
1,2,B E D
2,3,B E D
3,4,B E D
4,5,B E D


In [28]:
changed = 0

for i in tqdm(range(50)):

    row = test.iloc[i]

    original = predict_probabilities(
        deberta,
        deberta_tokenizer,
        row,
        tta=False
    )

    augmented = tta_prediction(
        deberta,
        deberta_tokenizer,
        row
    )

    if np.argmax(original) != np.argmax(augmented):

        changed += 1

  0%|          | 0/50 [00:00<?, ?it/s]

In [29]:
print("Q6")

print(changed)

Q6
0


In [30]:
different = 0

for i in tqdm(range(100)):

    row = test.iloc[i]

    d = predict_probabilities(
        deberta,
        deberta_tokenizer,
        row
    )

    r = predict_probabilities(
        roberta,
        roberta_tokenizer,
        row
    )

    e = weighted_ensemble(
        d,
        r
    )

    if np.argmax(d) != np.argmax(e):

        different += 1

  0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
print("Q7")

print(different)

Q7
0


In [32]:
gain = 0

for i in tqdm(range(100)):

    row = test.iloc[i]

    d = predict_probabilities(
        deberta,
        deberta_tokenizer,
        row
    )

    r = predict_probabilities(
        roberta,
        roberta_tokenizer,
        row
    )

    e = weighted_ensemble(
        d,
        r
    )

    if np.max(e) > np.max(d):

        gain += 1

  0%|          | 0/100 [00:00<?, ?it/s]

In [33]:
print("Q8")

print(gain)

Q8
0


In [34]:
changes = 0

for i in tqdm(range(100)):

    row = test.iloc[i]

    d = predict_probabilities(
        deberta,
        deberta_tokenizer,
        row
    )

    r = predict_probabilities(
        roberta,
        roberta_tokenizer,
        row
    )

    e = weighted_ensemble(
        d,
        r
    )

    d_top3 = topk_prediction(
        d
    )

    e_top3 = topk_prediction(
        e
    )

    if d_top3 != e_top3:

        changes += 1

  0%|          | 0/100 [00:00<?, ?it/s]

In [35]:
print("Q9")

print(changes)

Q9
84


In [36]:
actual = []

predictions = []

for i in tqdm(range(100)):

    row = train.iloc[i]

    d = predict_probabilities(
        deberta,
        deberta_tokenizer,
        row
    )

    r = predict_probabilities(
        roberta,
        roberta_tokenizer,
        row
    )

    e = weighted_ensemble(
        d,
        r
    )

    pred = topk_prediction(
        e,
        k=3
    )

    predictions.append(pred)

    actual.append(row["answer"])

  0%|          | 0/100 [00:00<?, ?it/s]

In [37]:
score = mapk(
    actual,
    predictions
)


print("Q10")

print(round(score,4))

Q10
0.4267


In [38]:
wandb.log({

    "Q1_Best_Option": id2label[q1_idx],
    "Q1_Probability": float(deberta_probs[q1_idx]),

    "Q2_Best_Option": id2label[q2_idx],

    "Q3_Best_Option": id2label[q3_idx],

    "Q4_Top3": prediction,

    "Q5_Total_Test_Rows": len(submission),

    "Q6_TTA_Changes": changed,

    "Q7_Top1_Differences": different,

    "Q8_Positive_Confidence_Gain": gain,

    "Q9_Top3_Changes": changes,

    "Q10_MAP3": float(score)

})

In [42]:
submission_artifact = wandb.Artifact(

    name="submission",

    type="dataset",

    description="Milestone 5 Kaggle Submission"

)

submission_artifact.add_file(
    "submission.csv"
)

wandb.log_artifact(
    submission_artifact
)

<Artifact submission>

In [43]:
results = pd.DataFrame({

    "Question":[
        "Q1",
        "Q2",
        "Q3",
        "Q4",
        "Q5",
        "Q6",
        "Q7",
        "Q8",
        "Q9",
        "Q10"
    ],

    "Answer":[
        f"{id2label[q1_idx]}, {round(float(deberta_probs[q1_idx]),4)}",
        id2label[q2_idx],
        id2label[q3_idx],
        prediction,
        len(submission),
        changed,
        different,
        gain,
        changes,
        round(score,4)
    ]

})

results.to_csv(
    "milestone5_results.csv",
    index=False
)

results.head()

,Question,Answer
0,Q1,"B, 0.2369"
1,Q2,B
2,Q3,B
3,Q4,B E D
4,Q5,500


In [44]:
results_artifact = wandb.Artifact(

    name="results",

    type="results"

)

results_artifact.add_file(
    "milestone5_results.csv"
)

wandb.log_artifact(
    results_artifact
)

<Artifact results>

In [45]:
wandb.summary["MAP@3"] = float(score)
wandb.summary["Submission Rows"] = len(submission)
wandb.summary["Top1 Changes"] = different
wandb.summary["Confidence Gain"] = gain
wandb.summary["Top3 Changes"] = changes
wandb.summary["TTA Changes"] = changed

In [46]:
wandb.finish()

Q10_MAP3,▁
Q1_Probability,▁
Q5_Total_Test_Rows,▁
Q6_TTA_Changes,▁
Q7_Top1_Differences,▁
Q8_Positive_Confidence_Gain,▁
Q9_Top3_Changes,▁
Confidence Gain,0
MAP@3,0.42667
Q10_MAP3,0.42667
Q1_Best_Option,B
